# Same-season calibration and development uncertainty

Calibration is evaluated only on earlier sessions from the same season. Offering-clustered bootstrap intervals show whether small point-estimate improvements are distinguishable from sampling noise.

In [ ]:
from __future__ import annotations
import hashlib, inspect, json, os, platform, sys
from pathlib import Path
import joblib, numpy as np, pandas as pd, sklearn
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import brier_score_loss, log_loss, roc_auc_score
from sklearn.pipeline import Pipeline

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT.name == 'v2' or not (PROJECT_ROOT / 'notebooks').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_ROOT = PROJECT_ROOT / 'data' / 'Enrollment-Data-master'
ARTIFACT_ROOT = PROJECT_ROOT / 'artifacts' / 'v2'
CACHE_ROOT = ARTIFACT_ROOT / 'cache'
MODEL_ROOT = PROJECT_ROOT / 'model'
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
CACHE_ROOT.mkdir(parents=True, exist_ok=True)
RANDOM_STATE = 20260812
SESSION_ORDER = ['20229','20235','20239','20245','20249','20255','20259','20265']
SEASONS = {'fall_winter':['20229','20239','20249','20259'], 'summer':['20235','20245','20255','20265']}
FINAL_TEST = {'fall_winter':'20259', 'summer':'20265'}
DEVELOPMENT = {k:[s for s in v if s != FINAL_TEST[k]] for k,v in SEASONS.items()}

def sha256(path: Path) -> str:
    h = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''): h.update(chunk)
    return h.hexdigest()

def fingerprint(payload) -> str:
    return hashlib.sha256(json.dumps(payload, sort_keys=True, default=str).encode()).hexdigest()[:16]

def versions():
    return {'python':platform.python_version(),'numpy':np.__version__,'pandas':pd.__version__,
            'scikit_learn':sklearn.__version__,'joblib':joblib.__version__}
BASE = ['position_to_capacity','waitlist_to_capacity','days_to_deadline','movement_3d','movement_7d','position','waitlist','capacity','capacity_changed_7d','position_to_waitlist','days_squared','log_waitlist','movement_velocity_7d']
CONTEXT = BASE + ['near_deadline_7d','days_under_7','days_under_14','days_over_60','position_ratio_near_7d','waitlist_ratio_near_7d','rank_over_30pct','campus_erin','campus_scar','term_winter','term_full_year','winter_near_7d','scar_near_7d']
RANK_MONOTONIC={'position_to_capacity':-1,'position':-1,'position_to_waitlist':-1,'position_ratio_near_7d':-1,'rank_over_30pct':-1}
def targeted(frame):
    f=frame.copy(); days=f.days_to_deadline.astype(float); near=(days<=7).astype('float32')
    f['near_deadline_7d']=near; f['days_under_7']=(7-days).clip(lower=0); f['days_under_14']=(14-days).clip(lower=0); f['days_over_60']=(days-60).clip(lower=0)
    f['position_ratio_near_7d']=f.position_to_capacity*near; f['waitlist_ratio_near_7d']=f.waitlist_to_capacity*near
    f['rank_over_30pct']=(f.position_to_capacity>.30).astype('float32'); f['campus_erin']=(f.campus=='ERIN').astype('float32'); f['campus_scar']=(f.campus=='SCAR').astype('float32')
    f['term_winter']=(f.term=='winter').astype('float32'); f['term_full_year']=(f.term=='full_year').astype('float32')
    f['winter_near_7d']=f.term_winter*near; f['scar_near_7d']=f.campus_scar*near
    return f
def make_model(features, *, leaf=15, l2=3.0):
    constraints=[RANK_MONOTONIC.get(x,0) for x in features]
    return Pipeline([('features',ColumnTransformer([('numeric',SimpleImputer(strategy='median'),features)],remainder='drop')),
      ('model',HistGradientBoostingClassifier(max_iter=200,learning_rate=.05,max_leaf_nodes=leaf,l2_regularization=l2,monotonic_cst=constraints,random_state=RANDOM_STATE))])
def ece(y,p,w,bins=10):
    edges=np.linspace(0,1,bins+1); ids=np.clip(np.digitize(p,edges)-1,0,bins-1); total=w.sum(); out=0
    for b in range(bins):
        m=ids==b
        if m.any(): out+=w[m].sum()/total*abs(np.average(y[m],weights=w[m])-np.average(p[m],weights=w[m]))
    return float(out)
def metrics(frame,p):
    y=frame.cleared.to_numpy(); w=frame.model_weight.to_numpy(); p=np.clip(np.asarray(p),1e-6,1-1e-6)
    auc=roc_auc_score(y,p,sample_weight=w) if np.unique(y).size>1 else np.nan
    return {'brier':brier_score_loss(y,p,sample_weight=w),'log_loss':log_loss(y,p,sample_weight=w,labels=[0,1]),'ece':ece(y,p,w),'auc':auc,'accuracy':np.average((p>=.5)==y,weights=w)}

from sklearn.linear_model import LogisticRegression


## Load the uniquely fingerprinted development decision

In [ ]:
cache_manifest=json.loads((ARTIFACT_ROOT/'cache-manifest.json').read_text()); current_cache_fingerprint=fingerprint(cache_manifest)
candidates=[]
for candidate_path in ARTIFACT_ROOT.glob('development-selection-*.json'):
    candidate=json.loads(candidate_path.read_text())
    if candidate.get('cache_fingerprint')==current_cache_fingerprint and 'paired_brier_differences_vs_base' in candidate: candidates.append((candidate_path,candidate))
if not candidates: raise RuntimeError('No development selection matches the current cache and paired-selection contract')
selection_path,selection=max(candidates,key=lambda item:item[0].stat().st_mtime_ns); spec=selection['spec']; features=spec['features']
session_samples={s:targeted(pd.read_pickle(CACHE_ROOT/f'{s}-positions-v2.pkl')) for sessions in DEVELOPMENT.values() for s in sessions}
assert selection['cache_fingerprint']==current_cache_fingerprint


## Same-season calibration experiment

In [ ]:
def logit(p): p=np.clip(p,1e-6,1-1e-6); return np.log(p/(1-p)).reshape(-1,1)
rows=[]; predictions=[]; calibration_parameters={}
for season,sessions in DEVELOPMENT.items():
    # Generate temporal out-of-fold predictions from expanding same-season fits.
    folds=[]
    for fold in range(1,len(sessions)):
        train=pd.concat([session_samples[s] for s in sessions[:fold]],ignore_index=True); valid=session_samples[sessions[fold]]
        model=make_model(**spec); model.fit(train[features],train.cleared,model__sample_weight=train.model_weight)
        folds.append({'session':sessions[fold],'frame':valid,'probability':model.predict_proba(valid[features])[:,1]})
    calibration_fold,validation_fold=folds[0],folds[-1]
    calibration_frame=calibration_fold['frame']; val=validation_fold['frame']; p_raw=validation_fold['probability']
    if calibration_frame.cleared.nunique()<2:
        p_platt=p_raw.copy()
    else:
        provisional=LogisticRegression(C=1e6,solver='lbfgs').fit(logit(calibration_fold['probability']),calibration_frame.cleared,sample_weight=calibration_frame.model_weight)
        p_platt=provisional.predict_proba(logit(p_raw))[:,1]
    for method,p in [('none',p_raw),('platt_same_season',p_platt)]: rows.append({'season':season,'method':method,**metrics(val,p)})
    predictions.append((season,val,p_raw,p_platt))
    all_oof_probability=np.concatenate([fold['probability'] for fold in folds]); all_oof=pd.concat([fold['frame'] for fold in folds],ignore_index=True)
    if all_oof.cleared.nunique()<2: raise RuntimeError(f'{season}: cannot fit calibration with one outcome class')
    final_platt=LogisticRegression(C=1e6,solver='lbfgs').fit(logit(all_oof_probability),all_oof.cleared,sample_weight=all_oof.model_weight)
    calibration_parameters[season]={'coefficient':float(final_platt.coef_[0,0]),'intercept':float(final_platt.intercept_[0]),'fit':'temporal_same_season_oof'}
calibration_scores=pd.DataFrame(rows); calibration_scores

## Offering-clustered bootstrap intervals

In [ ]:
def offering_errors(frame,p):
    work=pd.DataFrame({'offering_id':frame.offering_id.to_numpy(),'weight':frame.model_weight.to_numpy(),'error':frame.model_weight.to_numpy()*(frame.cleared.to_numpy()-np.asarray(p))**2})
    return work.groupby('offering_id',sort=False).agg(error=('error','sum'),weight=('weight','sum')).to_numpy().T
def bootstrap_from_offerings(error,weight,reps=1000):
    rng=np.random.default_rng(RANDOM_STATE); chosen=rng.integers(0,len(error),size=(reps,len(error)))
    values=error[chosen].sum(axis=1)/weight[chosen].sum(axis=1)
    return np.quantile(values,[.025,.5,.975]).tolist()
def clustered_bootstrap(frame,p,reps=1000): return bootstrap_from_offerings(*offering_errors(frame,p),reps)
def paired_clustered_bootstrap(frame,left,right,reps=1000):
    left_error,weight=offering_errors(frame,left); right_error,_=offering_errors(frame,right)
    return bootstrap_from_offerings(left_error-right_error,weight,reps)
uncertainty=[]
for season,val,p_raw,p_platt in predictions:
    uncertainty += [{'season':season,'method':'none','brier_ci':clustered_bootstrap(val,p_raw)},
                    {'season':season,'method':'platt_same_season','brier_ci':clustered_bootstrap(val,p_platt)},
                    {'season':season,'comparison':'platt_minus_none','paired_brier_difference_ci':paired_clustered_bootstrap(val,p_platt,p_raw)}]
uncertainty

## Lock calibration and release thresholds before latest-session evaluation

In [ ]:
paired_ci={item['season']:item['paired_brier_difference_ci'] for item in uncertainty if item.get('comparison')=='platt_minus_none'}
# Require the entire paired 95% interval to favor calibration.
calibration={season:('platt_same_season' if interval[2]<0 else 'none') for season,interval in paired_ci.items()}
RELEASE_THRESHOLDS={'oracle_brier_lt_literal':True,'ece_lte':.05,'near_deadline_gap_lte':.08,'large_queue_gap_lte':.08,'max_probability_gap_lte':.10}
locked={'selection_fingerprint':selection['fingerprint'],'spec':spec,'calibration':calibration,
 'calibration_parameters':calibration_parameters,'scores':calibration_scores.to_dict('records'),'uncertainty':uncertainty,'release_thresholds':RELEASE_THRESHOLDS,'versions':versions()}
locked['fingerprint']=fingerprint(locked)
for old_path in ARTIFACT_ROOT.glob('locked-spec-*.json'): old_path.unlink()
path=ARTIFACT_ROOT/f'locked-spec-{locked["fingerprint"]}.json'; path.write_text(json.dumps(locked,indent=2),encoding='utf-8')
locked, path

## Interpretation

Calibration is selected only when the paired offering-clustered interval favors it. Release thresholds are locked here before the latest completed Fall/Winter and Summer evaluation sessions are loaded.